# 07.1 — BERT Deep Dive

**BERT (Bidirectional Encoder Representations from Transformers)** changed NLP in 2018. Key idea: pre-train a Transformer encoder bidirectionally on unlabeled text, then fine-tune on downstream tasks.

**GPT vs BERT:**
- GPT: decoder-only, left-to-right (causal). Good for generation.
- BERT: encoder-only, bidirectional. Good for understanding (classification, NER, QA).

**Two pre-training objectives:**
1. **MLM (Masked Language Modeling):** mask 15% of tokens, predict them. Forces bidirectional context.
2. **NSP (Next Sentence Prediction):** given two sentences, predict if B follows A. (Later shown to be less useful.)

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random

# ---- BERT Architecture from scratch ----
# BERT-base: 12 layers, 768 hidden, 12 heads, 110M params
# BERT-large: 24 layers, 1024 hidden, 16 heads, 340M params
# We build a BERT-tiny for illustration.

class BERTEmbedding(nn.Module):
    """
    BERT has THREE embedding types summed together:
    1. Token embeddings (WordPiece vocab, ~30k)
    2. Segment embeddings (sentence A=0 or B=1)
    3. Position embeddings (learned, max 512)
    
    Unlike the original Transformer which uses sinusoidal PE,
    BERT uses learned positional embeddings.
    """
    def __init__(self, vocab_size, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.seg_emb = nn.Embedding(2, d_model)        # segment A or B
        self.pos_emb = nn.Embedding(max_len, d_model)  # learned, not sinusoidal
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    
    def forward(self, tokens, segments=None):
        B, T = tokens.shape
        positions = torch.arange(T, device=tokens.device).unsqueeze(0)
        if segments is None:
            segments = torch.zeros_like(tokens)  # all segment A
        x = self.tok_emb(tokens) + self.pos_emb(positions) + self.seg_emb(segments)
        return self.drop(self.norm(x))


class BERTSelfAttention(nn.Module):
    """Full bidirectional self-attention — no causal mask."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        B, T, D = x.shape
        def split(t):
            return t.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        Q, K, V = split(self.W_Q(x)), split(self.W_K(x)), split(self.W_V(x))
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.drop(F.softmax(scores, dim=-1))
        out = (attn @ V).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_O(out)


class BERTLayer(nn.Module):
    """BERT uses Post-LN (the original paper), unlike modern Pre-LN."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = BERTSelfAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),  # BERT uses GELU, not ReLU
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Post-LN: normalize AFTER the residual addition
        x = self.norm1(x + self.drop(self.attn(x, mask)))
        x = self.norm2(x + self.ff(x))
        return x


class BERTModel(nn.Module):
    """
    Minimal BERT encoder.
    Outputs: contextual representations for every token.
    The [CLS] token's representation is used for sentence-level tasks.
    """
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=4, 
                 d_ff=256, max_len=128, dropout=0.1):
        super().__init__()
        self.embedding = BERTEmbedding(vocab_size, d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            BERTLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.pool = nn.Linear(d_model, d_model)  # pooler for [CLS] token
    
    def forward(self, tokens, segments=None, mask=None):
        x = self.embedding(tokens, segments)
        for layer in self.layers:
            x = layer(x, mask)
        # [CLS] token is always at position 0
        cls_repr = torch.tanh(self.pool(x[:, 0]))  # BERT uses tanh on the pooler
        return x, cls_repr  # (all token reprs, sentence repr)


# Quick test
VOCAB = 1000
bert = BERTModel(vocab_size=VOCAB, d_model=64, n_heads=4, n_layers=4)
n_params = sum(p.numel() for p in bert.parameters())
print(f"Mini-BERT params: {n_params:,}")

# Simulate: [CLS] + sentence A + [SEP] + sentence B + [SEP]
tokens = torch.randint(1, VOCAB, (2, 20))   # batch=2, seq_len=20
segments = torch.zeros(2, 20, dtype=torch.long)
segments[:, 10:] = 1  # second sentence starts at position 10

all_repr, cls_repr = bert(tokens, segments)
print(f"Token representations: {all_repr.shape}")
print(f"[CLS] representation:  {cls_repr.shape}")
print(f"[CLS] repr used for: classification, NSP, sentence similarity")

In [ ]:
# ---- MLM: Masked Language Modeling ----
# 15% of tokens are selected for masking:
#   80% -> replaced with [MASK]
#   10% -> replaced with random token
#   10% -> kept unchanged
# Predict the original token at ALL selected positions.

def create_mlm_batch(token_ids: list[int], mask_token_id: int, 
                     vocab_size: int, mask_prob=0.15):
    """
    Returns (masked_input, labels) where:
    - labels[i] = original token if masked, else -100 (ignore in loss)
    - -100 is PyTorch's CrossEntropyLoss ignore_index
    """
    masked = token_ids.copy()
    labels = [-100] * len(token_ids)
    
    # Skip special tokens (0=PAD, 1=CLS, 2=SEP)
    candidate_positions = [i for i, t in enumerate(token_ids) if t > 2]
    n_mask = max(1, int(len(candidate_positions) * mask_prob))
    selected = random.sample(candidate_positions, min(n_mask, len(candidate_positions)))
    
    for pos in selected:
        labels[pos] = token_ids[pos]  # save original
        r = random.random()
        if r < 0.8:
            masked[pos] = mask_token_id  # 80%: replace with [MASK]
        elif r < 0.9:
            masked[pos] = random.randint(3, vocab_size - 1)  # 10%: random token
        # else: keep unchanged (10%)
    
    return masked, labels


# Example
VOCAB_SIZE = 100
CLS, SEP, MASK = 1, 2, 3

sentence = [CLS] + list(range(10, 30)) + [SEP]  # [CLS] + 20 tokens + [SEP]
masked_input, labels = create_mlm_batch(sentence, MASK, VOCAB_SIZE)

print("Original: ", sentence)
print("Masked:   ", masked_input)
print("Labels:   ", labels)
print()
n_masked = sum(1 for l in labels if l != -100)
print(f"Tokens masked: {n_masked}/{len(sentence)} = {n_masked/len(sentence):.1%}")
print()
print("Why 80/10/10 split?")
print("  Pure masking -> model learns [MASK] is special, never sees it at fine-tune time.")
print("  10% random -> model can't rely on context alone to ignore masked positions.")
print("  10% unchanged -> model must represent every token's identity, not just masked.")

In [ ]:
# ---- MLM Pre-training Head ----

class MLMHead(nn.Module):
    """
    Applied on top of BERT for MLM pre-training.
    Takes token representations -> predicts original token.
    """
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.dense = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.decoder = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        return self.decoder(self.norm(F.gelu(self.dense(x))))


class BERTForMLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=4):
        super().__init__()
        self.bert = BERTModel(vocab_size, d_model, n_heads, n_layers)
        self.mlm_head = MLMHead(d_model, vocab_size)
    
    def forward(self, tokens, segments=None, mask=None, mlm_labels=None):
        all_repr, _ = self.bert(tokens, segments, mask)
        logits = self.mlm_head(all_repr)  # [B, T, vocab_size]
        
        loss = None
        if mlm_labels is not None:
            # Only compute loss where label != -100
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                mlm_labels.view(-1),
                ignore_index=-100
            )
        return logits, loss


# Quick forward pass to verify shapes
V = 200
bert_mlm = BERTForMLM(vocab_size=V, d_model=64, n_heads=4, n_layers=2)

batch_tokens = torch.randint(4, V, (4, 16))   # [B=4, T=16]
batch_tokens[:, 0] = CLS
batch_tokens[:, -1] = SEP

# Create MLM labels: -100 everywhere except 2 random positions
batch_labels = torch.full_like(batch_tokens, -100)
batch_labels[:, 5] = batch_tokens[:, 5].clone()  # predict position 5
batch_labels[:, 9] = batch_tokens[:, 9].clone()  # predict position 9
batch_tokens[:, 5] = MASK
batch_tokens[:, 9] = MASK

logits, loss = bert_mlm(batch_tokens, mlm_labels=batch_labels)
print(f"Logits shape: {logits.shape}  (B, T, vocab)")
print(f"MLM loss: {loss.item():.4f}  (untrained, expect ~log({V})={math.log(V):.2f})")
print()
print("Key architecture differences from GPT:")
print("  GPT: causal mask -> each token only sees left context")
print("  BERT: no mask   -> each token sees ALL context (bidirectional)")
print("  This makes BERT better at understanding, GPT better at generation")

In [ ]:
# ---- Attention pattern analysis ----
# BERT's attention heads learn different roles:
#   - Some heads focus on adjacent tokens (local)
#   - Some heads focus on [CLS] or [SEP] (global anchors)
#   - Some heads mirror the attention (symmetric)
#   - Some heads encode syntactic relations (subject -> verb)

print("BERT attention head roles (from Clark et al. 2019):")
print()
head_behaviors = [
    ("Layer 2, Heads 0-2", "Attend to previous/next token (local)"),
    ("Layer 2, Head 4",    "Attend to [SEP] token (sentence boundary)"),
    ("Layer 5, Head 5",    "Attend to [CLS] token (global info)"),
    ("Layer 8, Head 10",   "Coreferent nouns attend to each other"),
    ("Layer 10, Head 0",   "Subjects attend to their verbs"),
]
for name, behavior in head_behaviors:
    print(f"  {name:25}: {behavior}")

print()
print("Why does BERT use [CLS]?")
print("  [CLS] is prepended to every input. It has NO semantic content of its own.")
print("  During pre-training (NSP), [CLS] must aggregate sentence-pair info.")
print("  After training, [CLS] representation = compressed sentence summary.")
print("  Fine-tune: attach a linear layer to [CLS] for classification tasks.")
print()
print("Note: RoBERTa removes NSP. Still uses [CLS] for classification.")
print("  NSP proved not helpful — the [CLS] learning comes mostly from MLM.")

## Summary

**BERT = Transformer encoder + bidirectional attention + MLM pre-training**

**Key design choices:**
- Three embeddings: token + segment + position (all learned)
- Post-LN (original) vs Pre-LN (modern)
- GELU activation (smoother than ReLU)
- [CLS] token for sentence-level representation
- 80/10/10 masking strategy for robustness

**What BERT is NOT good at:** Autoregressive generation. Its bidirectionality that makes it great for understanding makes it unsuitable for left-to-right text generation.

---
**Next:** 07.2 — Fine-tune BERT on downstream tasks